In [1]:
import os

BASE_PATH = "/kaggle/input/competitions/kaggle-hac-x-26-data-sprint-to-the-peak/synthetic_industrial_metal_surface_defects"

print(os.listdir(BASE_PATH))

['config.json', 'industrial_defect_dataset', 'README.md', 'metadata.csv']


In [2]:
BASE_PATH = "/kaggle/input/competitions/kaggle-hac-x-26-data-sprint-to-the-peak/synthetic_industrial_metal_surface_defects/industrial_defect_dataset"

TRAIN_DIR = BASE_PATH + "/train"
VAL_DIR = BASE_PATH + "/val"

In [3]:
import os

print("Train classes:", os.listdir(TRAIN_DIR))
print("Val classes:", os.listdir(VAL_DIR))

Train classes: ['normal', 'hole', 'rust', 'scratch', 'crack']
Val classes: ['normal', 'hole', 'rust', 'scratch', 'crack']


In [4]:
train_files = set(os.listdir(TRAIN_DIR / "crack"))
val_files = set(os.listdir(VAL_DIR / "crack"))

print("Overlap:", train_files.intersection(val_files))

TypeError: unsupported operand type(s) for /: 'str' and 'str'

In [ ]:
"""
High-Performance Industrial Defect Detection System
KaggleHacX '26 - Data Sprint to the Peak
Team: [YourTeamName]

Training pipeline using EfficientNetV2B0 transfer learning with TensorFlow/Keras.
"""

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
)
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight

# ─────────────────────────────────────────────
# 0. REPRODUCIBILITY
# ─────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Enable mixed precision for faster GPU training
mixed_precision.set_global_policy('mixed_float16')
print("Mixed precision policy:", mixed_precision.global_policy())

# ─────────────────────────────────────────────
# 1. CONFIGURATION
# ─────────────────────────────────────────────
BASE_DIR = Path("/kaggle/input/competitions/kaggle-hac-x-26-data-sprint-to-the-peak/"
                "synthetic_industrial_metal_surface_defects/industrial_defect_dataset")

TRAIN_DIR = BASE_DIR / "train"
VAL_DIR   = BASE_DIR / "val"

IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS_FROZEN    = 3    # Phase 1: frozen base (converges in 1-2 epochs on this dataset)
EPOCHS_FINETUNE  = 10   # Phase 2: fine-tune top layers
NUM_CLASSES = 5
CLASSES = ['crack', 'hole', 'normal', 'rust', 'scratch']

MODEL_SAVE_PATH = "model.keras"

print("TensorFlow version:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices('GPU')))

# ─────────────────────────────────────────────
# 2. DATA GENERATORS WITH AUGMENTATION
# ─────────────────────────────────────────────

def build_generators():
    """Build train and validation data generators."""

    train_datagen = ImageDataGenerator(
        preprocessing_function=preprocess_input,
        rotation_range=45,
        zoom_range=0.35,
        width_shift_range=0.25,
        height_shift_range=0.25,
        horizontal_flip=True,
        vertical_flip=True,
        brightness_range=[0.5, 1.5],
        shear_range=0.25,
        channel_shift_range=30,
        fill_mode='nearest',
    )

    val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

    train_gen = train_datagen.flow_from_directory(
        TRAIN_DIR,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=CLASSES,
        shuffle=True,
        seed=SEED,
    )

    val_gen = val_datagen.flow_from_directory(
        VAL_DIR,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=CLASSES,
        shuffle=False,
    )

    print(f"\nClass indices: {train_gen.class_indices}")
    print(f"Train samples: {train_gen.samples}  |  Val samples: {val_gen.samples}")
    return train_gen, val_gen


# ─────────────────────────────────────────────
# 3. CLASS WEIGHTS (handle imbalance)
# ─────────────────────────────────────────────

def compute_weights(train_gen):
    labels = train_gen.classes
    cw = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
    class_weight_dict = dict(enumerate(cw))
    print("\nClass weights:", class_weight_dict)
    return class_weight_dict


# ─────────────────────────────────────────────
# 4. MODEL ARCHITECTURE
# ─────────────────────────────────────────────

def build_model(freeze_base=True):
    """EfficientNetV2B0 + custom head with BatchNorm + Dropout."""

    # Load pretrained base (ImageNet weights)
    base = EfficientNetV2B0(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = not freeze_base

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    # EfficientNetV2 preprocessing is handled by preprocess_input in the generator
    x = base(inputs, training=not freeze_base)

    # Global Average Pooling
    x = layers.GlobalAveragePooling2D()(x)

    # Dense block 1
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='swish')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)

    # Dense block 2
    x = layers.Dense(256, activation='swish')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)

    # Output — explicit float32 for numerical stability with mixed precision
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.15),
        metrics=['accuracy']
    )

    return model


# ─────────────────────────────────────────────
# 5. CALLBACKS
# ─────────────────────────────────────────────

def get_callbacks(phase='frozen'):
    patience_es = 2 if phase == 'frozen' else 3
    return [
        EarlyStopping(monitor='val_accuracy', patience=patience_es, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-7, verbose=1),
        ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_accuracy', save_best_only=True, verbose=1),
    ]


# ─────────────────────────────────────────────
# 6. TRAINING — PHASE 1 (Frozen Base)
# ─────────────────────────────────────────────

def train_frozen(model, train_gen, val_gen, class_weight_dict):
    print("\n" + "="*60)
    print("PHASE 1: Training with frozen EfficientNetV2B0 base")
    print("="*60)

    history = model.fit(
        train_gen,
        epochs=EPOCHS_FROZEN,
        validation_data=val_gen,
        class_weight=class_weight_dict,
        callbacks=get_callbacks('frozen'),
        verbose=1,
    )
    return history


# ─────────────────────────────────────────────
# 7. TRAINING — PHASE 2 (Fine-Tune)
# ─────────────────────────────────────────────

def fine_tune(model, train_gen, val_gen, class_weight_dict):
    print("\n" + "="*60)
    print("PHASE 2: Fine-tuning EfficientNetV2B0 (last 50 layers)")
    print("="*60)

    # Partially unfreeze the base model — keep early layers frozen to avoid overfitting
    base_model = model.layers[1]
    base_model.trainable = True
    for layer in base_model.layers[:-50]:
        layer.trainable = False

    # Recompile with very low LR
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.15),
        metrics=['accuracy']
    )

    history = model.fit(
        train_gen,
        epochs=EPOCHS_FINETUNE,
        validation_data=val_gen,
        class_weight=class_weight_dict,
        callbacks=get_callbacks('finetune'),
        verbose=1,
    )
    return history


# ─────────────────────────────────────────────
# 8. EVALUATION
# ─────────────────────────────────────────────

def evaluate(model, val_gen):
    print("\n" + "="*60)
    print("EVALUATION ON VALIDATION SET")
    print("="*60)

    val_gen.reset()
    y_pred_probs = model.predict(val_gen, verbose=1)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = val_gen.classes

    acc = accuracy_score(y_true, y_pred)
    print(f"\nValidation Accuracy: {acc:.4f} ({acc*100:.2f}%)")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=CLASSES))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES)
    plt.title('Confusion Matrix — Validation Set')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=150)
    plt.close()
    print("Confusion matrix saved to confusion_matrix.png")

    return acc, y_pred, y_true


# ─────────────────────────────────────────────
# 9. PLOT TRAINING HISTORY
# ─────────────────────────────────────────────

def plot_history(hist1, hist2=None):
    acc = hist1.history['accuracy']
    val_acc = hist1.history['val_accuracy']
    loss = hist1.history['loss']
    val_loss = hist1.history['val_loss']

    if hist2:
        acc += hist2.history['accuracy']
        val_acc += hist2.history['val_accuracy']
        loss += hist2.history['loss']
        val_loss += hist2.history['val_loss']

    epochs_range = range(len(acc))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs_range, acc, label='Train Accuracy')
    axes[0].plot(epochs_range, val_acc, label='Val Accuracy')
    axes[0].set_title('Accuracy over Epochs')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(epochs_range, loss, label='Train Loss')
    axes[1].plot(epochs_range, val_loss, label='Val Loss')
    axes[1].set_title('Loss over Epochs')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True)

    plt.suptitle('EfficientNetV2B0 Training History', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150)
    plt.close()
    print("Training history saved to training_history.png")


# ─────────────────────────────────────────────
# 10. MAIN
# ─────────────────────────────────────────────

if __name__ == '__main__':
    # Build data generators
    train_gen, val_gen = build_generators()

    # Class weights for imbalance
    class_weight_dict = compute_weights(train_gen)

    # ---- Phase 1: Frozen base ----
    model = build_model(freeze_base=True)
    model.summary()

    history1 = train_frozen(model, train_gen, val_gen, class_weight_dict)

    # ---- Phase 2: Fine-tune ----
    history2 = fine_tune(model, train_gen, val_gen, class_weight_dict)

    # ---- Evaluation ----
    acc, y_pred, y_true = evaluate(model, val_gen)

    # ---- Save plots ----
    plot_history(history1, history2)

    # ---- Final model is already saved by ModelCheckpoint ----
    print(f"\n✅ Best model saved to: {MODEL_SAVE_PATH}")
    print(f"✅ Final Validation Accuracy: {acc*100:.2f}%")

In [ ]:
"""
Model Evaluation & Performance Metrics
KaggleHacX '26 | Team: [YourTeamName]

This script evaluates the trained model ('model.keras') on the validation/test dataset.
It generates:
  1. Accuracy, Precision, Recall, and F1-Scores (Macro & Weighted)
  2. A detailed Classification Report
  3. A beautifully styled Confusion Matrix visualization

Run on Kaggle after training.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# ─────────────────────────────────────────────
# 1. CONFIGURATION
# ─────────────────────────────────────────────
# Update BASE_DIR to your exact Kaggle path if it differs!
BASE_DIR = Path("/kaggle/input/competitions/kaggle-hac-x-26-data-sprint-to-the-peak/synthetic_industrial_metal_surface_defects/industrial_defect_dataset")
TEST_DIR = BASE_DIR / "val"  # Replace with 'test' if a separate test directory exists

MODEL_PATH = "model.keras"
IMG_SIZE = 224
BATCH_SIZE = 32
CLASSES = ['crack', 'hole', 'normal', 'rust', 'scratch']

print(f"Loading test data from: {TEST_DIR}")
if not TEST_DIR.exists():
    raise FileNotFoundError(f"Test directory not found: {TEST_DIR}\nPlease update BASE_DIR.")

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model file not found: {MODEL_PATH}\nPlease train the model first.")

# ─────────────────────────────────────────────
# 2. LOAD DATA
# ─────────────────────────────────────────────
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASSES,
    shuffle=False  # MUST be False to match predictions with true labels!
)

print(f"\nFound {test_gen.samples} images for evaluation.")

# ─────────────────────────────────────────────
# 3. RUN PREDICTIONS
# ─────────────────────────────────────────────
print("\nLoading model and running predictions...")
model = keras.models.load_model(MODEL_PATH)

# Get predictions
preds_prob = model.predict(test_gen, verbose=1)
preds_classes = np.argmax(preds_prob, axis=1)

# Get true labels
true_classes = test_gen.classes

# ─────────────────────────────────────────────
# 4. CALCULATE METRICS
# ─────────────────────────────────────────────
accuracy = accuracy_score(true_classes, preds_classes)
f1_macro = f1_score(true_classes, preds_classes, average='macro')
f1_weighted = f1_score(true_classes, preds_classes, average='weighted')

print("\n" + "="*50)
print("🏆 MODEL PERFORMANCE SUMMARY")
print("="*50)
print(f"Accuracy:        {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"F1-Score (Macro): {f1_macro:.4f}  <-- Balances all classes equally")
print(f"F1-Score (Wght):  {f1_weighted:.4f}")
print("="*50)

print("\n📊 DETAILED CLASSIFICATION REPORT")
print(classification_report(true_classes, preds_classes, target_names=CLASSES, digits=4))

# ─────────────────────────────────────────────
# 5. VISUALIZE CONFUSION MATRIX
# ─────────────────────────────────────────────
def plot_confusion_matrix(y_true, y_pred, classes, save_path="confusion_matrix.png"):
    cm = confusion_matrix(y_true, y_pred)
    
    # Setup plot styling
    plt.figure(figsize=(10, 8))
    sns.set_theme(style="white")
    
    # Create heatmap
    ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                     xticklabels=classes, yticklabels=classes,
                     linewidths=1, linecolor='white',
                     annot_kws={"size": 14, "weight": "bold"})
    
    # Styling
    plt.title('Defect Detection Confusion Matrix', fontsize=18, fontweight='bold', pad=20)
    plt.ylabel('True Class', fontsize=14, fontweight='bold', labelpad=10)
    plt.xlabel('Predicted Class', fontsize=14, fontweight='bold', labelpad=10)
    
    # Rotate tick labels
    plt.xticks(rotation=45, ha='right', fontsize=12)
    plt.yticks(rotation=0, fontsize=12)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"\n✅ Confusion matrix visualization saved as '{save_path}'")
    plt.show()

plot_confusion_matrix(true_classes, preds_classes, CLASSES)

# Save metrics to a text file for submission/records
with open("evaluation_metrics.txt", "w") as f:
    f.write("Industrial Defect Detection - Performance Metrics\n")
    f.write("="*50 + "\n")
    f.write(f"Accuracy:        {accuracy:.4f}\n")
    f.write(f"F1-Score (Macro): {f1_macro:.4f}\n")
    f.write(f"F1-Score (Wght):  {f1_weighted:.4f}\n\n")
    f.write("Classification Report:\n")
    f.write(classification_report(true_classes, preds_classes, target_names=CLASSES, digits=4))

print("✅ Evaluation metrics saved to 'evaluation_metrics.txt'")